In [ ]:
import pandas as pd

Question - 1

In [3]:
import pandas as pd

roll_no = "1024170157"

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

category_map = {
    0: "billing",
    1: "account",
    2: "general"
}

digits = roll_no[-2:]

personalized_entries = []

questions = {
    "billing": (
        "how can i check my pending fee",
        "You can check pending fees in the payment portal.",
        "fee payment pending billing"
    ),
    "account": (
        "how do i update my registered mobile number",
        "Go to Account Settings and update your mobile number.",
        "account mobile number update"
    ),
    "general": (
        "how can i contact customer support",
        "You can contact support through email or the helpdesk.",
        "support help contact assistance"
    )
}

for digit in digits:
    category = category_map[int(digit) % 3]

    question, answer, keywords = questions[category]

    personalized_entries.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })

faq_data = pd.DataFrame(fixed_entries + personalized_entries)

In [4]:
faq_data

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i contact customer support,You can contact support through email or the h...,support help contact assistance,general
5,how do i update my registered mobile number,Go to Account Settings and update your mobile ...,account mobile number update,account


Question - 2

In [5]:
def score_hypothesis(query, df):
    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():
        keywords = set(row['keywords'].lower().split())

        matched = query_words.intersection(keywords)

        score = len(matched) / len(keywords)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": round(score * 100, 2)
            })

    results = sorted(
        results,
        key=lambda x: x["confidence"],
        reverse=True
    )

    return pd.DataFrame(results)


query = "how can i pay fee"

print(score_hypothesis(query, faq_data))

                 question                                      answer  \
0   how can i pay the fee  You can pay via UPI, card, or net banking.   
1  what is the annual fee                   The annual fee is Rs 500.   

  category  confidence  
0  billing        50.0  
1  billing        25.0  


Question - 3

In [7]:
def same_category(category_name, df):
    return df[df['category'] == category_name]


category = faq_data.iloc[4]['category']

print("Selected Category:", category)

print(same_category(category, faq_data))

Selected Category: general
                             question  \
2         what are your working hours   
4  how can i contact customer support   

                                              answer  \
2                          We are open 9 AM to 5 PM.   
4  You can contact support through email or the h...   

                          keywords category  
2           hours timing open time  general  
4  support help contact assistance  general  


Question - 4

In [8]:
index = 0

print("Selected FAQ:", faq_data.loc[index, 'question'])

new_keyword = input("Enter a new keyword: ")

faq_data.loc[index, 'keywords'] = (
    faq_data.loc[index, 'keywords'] + " " + new_keyword
)

filename = "1024170157_faq_data.csv"

faq_data.to_csv(filename, index=False)

print("Updated DataFrame:")
print(faq_data)

print("File saved as:", filename)

Selected FAQ: what is the annual fee
Enter a new keyword: contact
Updated DataFrame:
                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4           how can i contact customer support   
5  how do i update my registered mobile number   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can contact support through email or the h...   
5  Go to Account Settings and update your mobile ...   

                          keywords category  
0    fee cost price charge contact  billing  
1             password reset login  account  
2           hours timing open tim

Question - 5

In [9]:
category_counts = faq_data.groupby('category').size()

print(category_counts)

category
account    2
billing    2
general    2
dtype: int64


Question - 6

In [10]:
def score_hypothesis_with_ties(query, df):

    results = score_hypothesis(query, df)

    if results.empty:
        print("No matching FAQ found.")
        return results

    max_score = results['confidence'].max()

    top_matches = results[
        results['confidence'] == max_score
    ]

    if len(top_matches) > 1:
        print("Tie detected! Multiple FAQs have the highest score:")
    else:
        print("Best matching FAQ:")

    print(top_matches.to_string(index=False))

    return results

In [11]:
score_hypothesis_with_ties("fee", faq_data)

Best matching FAQ:
             question                                     answer category  confidence
how can i pay the fee You can pay via UPI, card, or net banking.  billing        25.0


,question,answer,category,confidence
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,25.0
1,what is the annual fee,The annual fee is Rs 500.,billing,20.0


In [12]:
score_hypothesis_with_ties("password reset login", faq_data)

Best matching FAQ:
             question                           answer category  confidence
how to reset password Go to Settings > Reset Password.  account       100.0


,question,answer,category,confidence
0,how to reset password,Go to Settings > Reset Password.,account,100.0
